# Feature Engineering — Complete Cheatsheet (All-in-One)

This single notebook merges **all 5 sections** of Feature Engineering:

1. **Data Distribution + Train/Test Split** 
2. **Missing Value Handling** (CCA, Mean/Median, Arbitrary, Random Sample, Frequent value, SimpleImputer) 
3. **Outlier Detection & Handling** (Z-Score, IQR, Winsorization) 
4. **Encoding** (Ordinal, OneHot, Label) 
5. **Scaling** (Standard, MinMax, Robust) 
6. **ColumnTransformer** (clean end-to-end pipeline)

**Data files used:**
- `titanic_data_updated.csv` — main Titanic dataset
- `marks_dataset.csv` — used for Winsorization & RobustScaler demos
- `science_job.csv` — used for CCA demo

> Make sure these CSVs are in the same folder as this notebook (or update the paths).

## 0. Setup — All Imports in One Cell

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    OrdinalEncoder, OneHotEncoder, LabelEncoder,
    StandardScaler, MinMaxScaler, RobustScaler
)
from sklearn.compose import ColumnTransformer

pd.set_option('display.max_columns', None)

---
# 1. Data Distribution + Train / Test Split

**Why split FIRST (before any imputation / scaling)?** 
To prevent **data leakage**. We learn statistics (mean, median, mode, min, max, etc.) from the **train set only**, then apply them to the test set.

In [ ]:
df = pd.read_csv('1. Data Distribution-splitting-missing values/titanic_data_updated.csv')
df.head()

In [ ]:
# Drop columns that are pure identifiers / free text (not useful for ML directly)
df.drop(['PassengerId', 'Name', 'Ticket'], axis=1, inplace=True)
df.head()

In [ ]:
# % of missing values per column
df.isnull().mean() * 100

In [ ]:
# Feature & target separation
X = df.drop(['Survived'], axis=1)
y = df['Survived']

# Train-test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print('Train shape:', X_train.shape, '| Test shape:', X_test.shape)

In [ ]:
# Check missing values in train / test separately
print('--- X_train missing ---')
print(X_train.isnull().sum())
print('\n--- X_test missing ---')
print(X_test.isnull().sum())

---
# 2. Missing Value Handling

| Technique | When to use |
|---|---|
| **CCA** (drop rows) | Missing < 5% AND missing is random |
| **Mean imputation** | Numerical + roughly normal distribution |
| **Median imputation** | Numerical + skewed / outliers present |
| **Arbitrary value** (-1, 99, 999) | Tree models, want to flag missingness |
| **Random sample** | Preserve distribution exactly |
| **Mode / Most-frequent** | Categorical, few missing |
| **Constant 'Missing' + indicator** | Categorical, lots of missing (e.g., Cabin) |

## 2.1 CCA — Complete Case Analysis
Drop rows that have any NaN in selected columns. Only safe when missing < 5%.

In [ ]:
# CCA demo on a separate dataset
try:
    cca_df = pd.read_csv('1. Data Distribution-splitting-missing values/science_job.csv')
    # Columns where missing < 5% and > 0%
    cca_cols = [c for c in cca_df.columns
                if 0 < cca_df[c].isnull().mean() < 0.05]
    print('CCA-safe columns:', cca_cols)

    new_df = cca_df[cca_cols].dropna()
    print('Original shape:', cca_df.shape, '| After CCA:', new_df.shape)
    print('Rows retained:', round(len(new_df)/len(cca_df) * 100, 2), '%')
except FileNotFoundError:
    print('science_job.csv not found — skipping CCA demo.')

**Always check distribution before & after CCA** to make sure we didn't introduce bias.

In [ ]:
# Numerical column — compare histograms / KDE
try:
    fig, ax = plt.subplots()
    cca_df['training_hours'].plot.density(color='red', ax=ax, label='Original')
    new_df['training_hours'].plot.density(color='green', ax=ax, label='After CCA')
    ax.legend(); plt.title('CCA — distribution check')
    plt.show()

    # Categorical column — ratio of categories
    print('Original ratio:\n', cca_df['enrolled_university'].value_counts(normalize=True))
    print('\nAfter CCA ratio:\n', new_df['enrolled_university'].value_counts(normalize=True))
except NameError:
    pass

## 2.2 Numerical — Mean / Median (pandas way)

In [ ]:
# Mean impute — fit on TRAIN, apply to both
age_mean = X_train['Age'].mean()
X_train['age_mean_imputed']   = X_train['Age'].fillna(age_mean)
X_test['age_mean_imputed']    = X_test['Age'].fillna(age_mean)

# Median impute
age_median = X_train['Age'].median()
X_train['age_median_imputed'] = X_train['Age'].fillna(age_median)
X_test['age_median_imputed']  = X_test['Age'].fillna(age_median)

X_train[['Age', 'age_mean_imputed', 'age_median_imputed']].head()

In [ ]:
# Visual check — original vs imputed distributions
fig, ax = plt.subplots()
X_train['Age'].plot(kind='kde', ax=ax, color='blue',  label='Original')
X_train['age_mean_imputed'].plot(kind='kde', ax=ax, color='green', label='Mean')
X_train['age_median_imputed'].plot(kind='kde', ax=ax, color='red',   label='Median')
ax.legend(); plt.title('Mean vs Median Imputation'); plt.show()

## 2.3 Numerical — Arbitrary Value Imputation
Useful for tree-based models when you want to flag missingness with a sentinel value.

In [ ]:
X_train['Age_99']     = X_train['Age'].fillna(99)
X_train['Age_minus1'] = X_train['Age'].fillna(-1)
X_train[['Age', 'Age_99', 'Age_minus1']].head()

## 2.4 Numerical — Random Sample Imputation
Best for preserving the original distribution exactly.

In [ ]:
X_train['Age_random'] = X_train['Age']
mask = X_train['Age_random'].isnull()

X_train.loc[mask, 'Age_random'] = (
    X_train['Age'].dropna()
                  .sample(mask.sum(), random_state=2)
                  .values
)

fig, ax = plt.subplots()
X_train['Age'].plot(kind='kde', ax=ax, color='blue', label='Original')
X_train['Age_random'].plot(kind='kde', ax=ax, color='red', label='Random Sample')
ax.legend(); plt.title('Random Sample Imputation'); plt.show()

## 2.5 Numerical — SimpleImputer (sklearn way)
Recommended for production / pipelines.

In [ ]:
# Drop the demo columns we added so they don't pollute the working dataframe
for c in ['age_mean_imputed', 'age_median_imputed', 'Age_99', 'Age_minus1', 'Age_random']:
    if c in X_train.columns: X_train.drop(c, axis=1, inplace=True)
    if c in X_test.columns:  X_test.drop(c, axis=1, inplace=True)

age_imputer = SimpleImputer(missing_values=np.nan, strategy='mean')
age_imputer.fit(X_train[['Age']])  # learn ONLY from train

X_train['Age'] = age_imputer.transform(X_train[['Age']]).ravel()
X_test['Age']  = age_imputer.transform(X_test[['Age']]).ravel()

print('Train NaNs remaining in Age:', X_train['Age'].isnull().sum())
print('Test  NaNs remaining in Age:', X_test['Age'].isnull().sum())

## 2.6 Categorical — Mode / Most-Frequent (for Embarked)

In [ ]:
embarked_imputer = SimpleImputer(missing_values=np.nan, strategy='most_frequent')
embarked_imputer.fit(X_train[['Embarked']])

X_train['Embarked'] = embarked_imputer.transform(X_train[['Embarked']]).ravel()
X_test['Embarked']  = embarked_imputer.transform(X_test[['Embarked']]).ravel()

print('Embarked NaNs after imputation:',
      X_train['Embarked'].isnull().sum(), X_test['Embarked'].isnull().sum())

## 2.7 Categorical — Constant `'Missing'` + Indicator (for Cabin)
Cabin has ~77% missing → can't use mode. Treat 'Missing' as its own category and add a boolean indicator.

In [ ]:
cabin_imputer = SimpleImputer(
    missing_values=np.nan,
    strategy='constant',
    fill_value='Missing',
    add_indicator=True
)
cabin_imputer.fit(X_train[['Cabin']])

X_train[['Cabin', 'cabin_missing_indicator']] = cabin_imputer.transform(X_train[['Cabin']])
X_test[['Cabin',  'cabin_missing_indicator']] = cabin_imputer.transform(X_test[['Cabin']])

X_train.head()

In [ ]:
# Verify zero NaNs remain
print(X_train.isnull().sum())
print('\n', X_test.isnull().sum())

---
# 3. Outlier Detection & Handling

| Method | When to use |
|---|---|
| **Z-Score** | Data is approximately normal (Gaussian) |
| **IQR** | Data is skewed / non-normal |
| **Winsorization (percentile clip)** | Custom percentile-based capping, very flexible |

## 3.1 Z-Score Method
Outlier if `|z| > 3` → roughly outside 3 standard deviations.

In [ ]:
# Z-score for Age
mean_age, std_age = X_train['Age'].mean(), X_train['Age'].std()
X_train['zscore_Age'] = (X_train['Age'] - mean_age) / std_age

outliers_age = X_train[abs(X_train['zscore_Age']) > 3]
print('Age outliers (|z|>3):', len(outliers_age))
outliers_age.head()

In [ ]:
# Z-score for Fare
mean_fare, std_fare = X_train['Fare'].mean(), X_train['Fare'].std()
X_train['zscore_Fare'] = (X_train['Fare'] - mean_fare) / std_fare

outliers_fare = X_train[abs(X_train['zscore_Fare']) > 3]
print('Fare outliers (|z|>3):', len(outliers_fare))

## 3.2 IQR Method
`Q1 - 1.5*IQR  ≤  value  ≤  Q3 + 1.5*IQR`

In [ ]:
# IQR for Age
age_Q1 = X_train['Age'].quantile(0.25)
age_Q3 = X_train['Age'].quantile(0.75)
age_IQR = age_Q3 - age_Q1
age_min = age_Q1 - 1.5 * age_IQR
age_max = age_Q3 + 1.5 * age_IQR
print(f'Age IQR bounds: [{age_min}, {age_max}]')

age_outlier_iqr = X_train[(X_train['Age'] < age_min) | (X_train['Age'] > age_max)]
print('Age outliers (IQR):', len(age_outlier_iqr))

In [ ]:
# IQR for Fare (clip at max(0, ...) because Fare cannot be negative)
fare_Q1 = X_train['Fare'].quantile(0.25)
fare_Q3 = X_train['Fare'].quantile(0.75)
fare_IQR = fare_Q3 - fare_Q1
fare_min = max(0, fare_Q1 - 1.5 * fare_IQR)
fare_max = fare_Q3 + 1.5 * fare_IQR
print(f'Fare IQR bounds: [{fare_min}, {fare_max}]')

fare_outlier_iqr = X_train[(X_train['Fare'] < fare_min) | (X_train['Fare'] > fare_max)]
print('Fare outliers (IQR):', len(fare_outlier_iqr))

### Apply: Z-Score trimming for Age + IQR clipping for Fare
Different columns can use different strategies depending on their distribution shape.

In [ ]:
# 1) Age — z-score trim (keep |z|<=3)
X_train = X_train[abs(X_train['zscore_Age']) <= 3]

# 2) Fare — IQR clip (cap extremes instead of dropping)
X_train['Fare'] = X_train['Fare'].clip(fare_min, fare_max)

# Cleanup helper columns
X_train.drop(['zscore_Age', 'zscore_Fare'], axis=1, inplace=True)
X_train.head()

## 3.3 Winsorization (Percentile-based Clipping)
Clip everything below the `p`-th percentile and above the `(100-p)`-th percentile.

In [ ]:
try:
    marks = pd.read_csv('2. Outlier(Z-score,IQR,Winsorization_Technique)/marks_dataset.csv')
    print(marks['maths_marks'].describe())

    sns.boxplot(data=marks, x='maths_marks'); plt.title('Before winsorization'); plt.show()

    # Clip at 12th & 88th percentile
    pct = 12 / 100
    lo = marks['maths_marks'].quantile(pct)
    hi = marks['maths_marks'].quantile(1 - pct)
    print(f'Winsorize bounds: [{lo}, {hi}]')

    marks['maths_marks'] = marks['maths_marks'].clip(lo, hi)
    sns.boxplot(data=marks, x='maths_marks'); plt.title('After winsorization'); plt.show()
    print(marks['maths_marks'].describe())
except FileNotFoundError:
    print('marks_dataset.csv not found — skipping Winsorization demo.')

---
# 4. Encoding Categorical Features

| Encoder | Use for | Output |
|---|---|---|
| **OrdinalEncoder** | Ordered categories (`first < second < third`) | Single int column |
| **OneHotEncoder** | Nominal (no order) — Sex, Embarked, etc. | One binary column per category |
| **LabelEncoder** | Target column `y` only | Single int column |

## 4.1 OrdinalEncoder — `Pclass` (third < second < first)

In [ ]:
pclass_encoder = OrdinalEncoder(categories=[['third', 'second', 'first']])
pclass_encoder.fit(X_train[['Pclass']])

X_train['encoded_Pclass'] = pclass_encoder.transform(X_train[['Pclass']]).ravel()
X_test['encoded_Pclass']  = pclass_encoder.transform(X_test[['Pclass']]).ravel()
X_train[['Pclass', 'encoded_Pclass']].head()

## 4.2 OneHotEncoder — `Embarked`

In [ ]:
embarked_ohe = OneHotEncoder(sparse_output=False).set_output(transform='pandas')
embarked_ohe.fit(X_train[['Embarked']])

X_train = pd.concat([X_train, embarked_ohe.transform(X_train[['Embarked']])], axis=1)
X_test  = pd.concat([X_test,  embarked_ohe.transform(X_test[['Embarked']])],  axis=1)
X_train.head()

## 4.3 OneHotEncoder with feature engineering — `Cabin_Deck`
Cabin values are codes like `C85`, `B42`. Use only the **first letter** (deck) as the category and drop the first one to avoid the dummy-variable trap.

In [ ]:
X_train['Cabin_Deck'] = X_train['Cabin'].astype(str).str[0]
X_test['Cabin_Deck']  = X_test['Cabin'].astype(str).str[0]

deck_ohe = OneHotEncoder(sparse_output=False, drop='first').set_output(transform='pandas')
deck_ohe.fit(X_train[['Cabin_Deck']])

X_train = pd.concat([X_train, deck_ohe.transform(X_train[['Cabin_Deck']])], axis=1)
X_test  = pd.concat([X_test,  deck_ohe.transform(X_test[['Cabin_Deck']])],  axis=1)
X_train.head()

In [ ]:
# Drop now-redundant original categorical / cabin columns
for col in ['Pclass', 'Sex', 'Embarked', 'Cabin', 'Cabin_Deck']:
    if col in X_train.columns: X_train.drop(col, axis=1, inplace=True)
    if col in X_test.columns:  X_test.drop(col, axis=1, inplace=True)
X_train.head()

## 4.4 LabelEncoder — for the TARGET only (`y_train`, `y_test`)

In [ ]:
le = LabelEncoder()
le.fit(y_train)

y_train = pd.Series(le.transform(y_train), index=y_train.index, name=y_train.name)
y_test  = pd.Series(le.transform(y_test),  index=y_test.index,  name=y_test.name)

# After removing outlier rows from X_train above, align y_train to remaining indices
y_train = y_train.loc[X_train.index]
y_train.head()

---
# 5. Feature Scaling

| Scaler | Formula | Use when |
|---|---|---|
| **StandardScaler** (Z-score) | `(x - mean) / std` | Data approximately normal |
| **MinMaxScaler** | `(x - min) / (max - min)` → [0, 1] | Bounded range needed, no extreme outliers |
| **RobustScaler** | `(x - median) / IQR` | Data has outliers |

## 5.1 StandardScaler — `Age` (roughly normal)

In [ ]:
ss = StandardScaler()
ss.fit(X_train[['Age']])

X_train['Age'] = ss.transform(X_train[['Age']]).ravel()
X_test['Age']  = ss.transform(X_test[['Age']]).ravel()

print('Mean after scaling:', round(X_train['Age'].mean(), 5))
print('Std  after scaling:', round(X_train['Age'].std(),  5))

## 5.2 MinMaxScaler — `Fare` (already clipped, want [0,1])

In [ ]:
mms = MinMaxScaler()
mms.fit(X_train[['Fare']])

X_train['Fare'] = mms.transform(X_train[['Fare']]).ravel()
X_test['Fare']  = mms.transform(X_test[['Fare']]).ravel()

X_train['Fare'].describe()

## 5.3 RobustScaler — when outliers are present (demo on marks dataset)

In [ ]:
try:
    marks2 = pd.read_csv('2. Outlier(Z-score,IQR,Winsorization_Technique)/marks_dataset.csv')
    print('Before RobustScaler:\n', marks2['maths_marks'].describe(), '\n')

    rs = RobustScaler()
    rs.fit(marks2[['maths_marks']])
    marks2['maths_marks'] = rs.transform(marks2[['maths_marks']]).ravel()

    print('After RobustScaler:\n', marks2['maths_marks'].describe())
    sns.kdeplot(data=marks2, x='maths_marks'); plt.title('RobustScaler output'); plt.show()
except FileNotFoundError:
    print('marks_dataset.csv not found — skipping RobustScaler demo.')

---
# 6. End-to-End Pipeline with `ColumnTransformer`

Everything above — imputation, encoding, scaling — combined cleanly in **one place**, applied **only after train/test split** to prevent data leakage. This is the **production-ready** pattern.

In [ ]:
# ---- Fresh start ----
df = pd.read_csv('1. Data Distribution-splitting-missing values/titanic_data_updated.csv')
df.drop(['PassengerId', 'Name', 'Ticket'], axis=1, inplace=True)

# Feature engineering: FamilySize = SibSp + Parch + 1
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

X = df.drop(['Survived'], axis=1)
y = df['Survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

### Step 1 — Imputation via ColumnTransformer

In [ ]:
imputer_transformer = ColumnTransformer(
    transformers=[
        ('age',      SimpleImputer(strategy='mean'),          ['Age']),
        ('embarked', SimpleImputer(strategy='most_frequent'), ['Embarked']),
        ('cabin',    SimpleImputer(strategy='constant',
                                   fill_value='Missing',
                                   add_indicator=True),       ['Cabin']),
    ],
    remainder='passthrough',
    verbose_feature_names_out=False,
)
imputer_transformer.set_output(transform='pandas')

imputer_transformer.fit(X_train)
X_train = imputer_transformer.transform(X_train)
X_test  = imputer_transformer.transform(X_test)
X_train.head()

### Step 2 — Outlier handling (Age z-score trim, Fare IQR clip)

In [ ]:
# Age — z-score trim
mean_age, std_age = X_train['Age'].mean(), X_train['Age'].std()
X_train['zscore_Age'] = (X_train['Age'] - mean_age) / std_age
X_train = X_train[abs(X_train['zscore_Age']) <= 3]
X_train.drop(['zscore_Age'], axis=1, inplace=True)

# Keep y_train aligned with the trimmed X_train rows
y_train = y_train.loc[X_train.index]

# Fare — IQR clip
fQ1 = X_train['Fare'].quantile(0.25)
fQ3 = X_train['Fare'].quantile(0.75)
fIQR = fQ3 - fQ1
fmin = max(0, fQ1 - 1.5 * fIQR)
fmax = fQ3 + 1.5 * fIQR
X_train['Fare'] = X_train['Fare'].clip(fmin, fmax)
X_test['Fare']  = X_test['Fare'].clip(fmin, fmax)  # apply same bounds to test

### Step 3 — Feature engineering (Cabin → Cabin_Deck)

In [ ]:
X_train['Cabin_Deck'] = X_train['Cabin'].astype(str).str[0]
X_test['Cabin_Deck']  = X_test['Cabin'].astype(str).str[0]

### Step 4 — Encoding + Scaling via ColumnTransformer (one shot)

In [ ]:
encoder_scaler = ColumnTransformer(
    transformers=[
        ('pclass',       OrdinalEncoder(categories=[['third', 'second', 'first']]),
                         ['Pclass']),
        ('embarked_sex', OneHotEncoder(sparse_output=False, drop='first',
                                       handle_unknown='ignore'),
                         ['Embarked', 'Sex', 'Cabin_Deck']),
        ('age_scaler',   StandardScaler(),
                         ['Age']),
        ('fare_scaler',  MinMaxScaler(),
                         ['Fare', 'FamilySize']),
    ],
    remainder='passthrough',
    verbose_feature_names_out=False,
)
encoder_scaler.set_output(transform='pandas')

encoder_scaler.fit(X_train)
X_train = encoder_scaler.transform(X_train)
X_test  = encoder_scaler.transform(X_test)

# Drop now-redundant raw columns that came through 'passthrough'
for col in ['Cabin', 'SibSp', 'Parch']:
    if col in X_train.columns: X_train.drop(col, axis=1, inplace=True)
    if col in X_test.columns:  X_test.drop(col, axis=1, inplace=True)

# Encode target
le = LabelEncoder()
le.fit(y_train)
y_train = pd.Series(le.transform(y_train), index=y_train.index, name='Survived')
y_test  = pd.Series(le.transform(y_test),  index=y_test.index,  name='Survived')

### Final, ML-ready dataset

In [ ]:
print('Final X_train:', X_train.shape, '| X_test:', X_test.shape)
X_train.head()

In [ ]:
X_test.head()

---
## Quick-reference summary

**Order to follow in every project:**
1. Load + drop irrelevant columns + check `isnull().mean()`
2. `train_test_split` FIRST
3. **Impute** missing values (fit on train, transform train + test)
4. **Outlier handling** (z-score for normal cols, IQR for skewed cols, Winsorization for percentile-based)
5. **Encoding** (Ordinal → ordered, OneHot → nominal, Label → target only)
6. **Scaling** (Standard → normal, MinMax → bounded, Robust → outliers)
7. Combine everything via `ColumnTransformer` for production
